# Modelo ordinal de situación jurídica

**Autora:** Laura Sofía Gómez Rosales

Este notebook es la interfaz de análisis actualizada. La lógica reutilizable y testeada vive en
`src/`; aquí se cargan los datos, se entrena el modelo ordinal y se muestran las métricas y figuras
del documento de trabajo.

Antes de ejecutarlo, genera los insumos con:

```bash
make download
make analysis
```


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from analysis import (
    OrdinalLogisticRegression,
    TARGET_ORDER,
    build_features,
    evaluate_model,
)
from cleaning import read_raw, snapshot, weighted_counts
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', None)


## 1. Corte actual y composición del stock

Se usa el corte de **2026-10-09**. La unidad de análisis es el stock de personas detenidas activas;
no se suman los cortes mensuales.


In [ ]:
frame = read_raw(ROOT / 'data' / 'raw' / 'colombianos_detenidos.csv')
current = snapshot(frame, '2026-10-09')

print(f'Filas del corte: {len(current):,}')
print(f'Detenidos activos: {current["CANTIDAD"].sum():,}')
print()
print('Países con más detenidos activos:')
display(weighted_counts(current, 'PAIS').head(10).to_frame())
print()
print('Delitos más reportados:')
display(weighted_counts(current, 'DELITO_LIMPIO').head(10).to_frame())


## 2. Entrenamiento y validación temporal

- Entrenamiento: corte 2025-11-01.
- Prueba: corte 2026-10-09.
- Variable objetivo ordenada: `EN INVESTIGACIÓN` -> `EN JUICIO` -> `CONDENADO`.
- `CANTIDAD` se usa como peso muestral.


In [ ]:
train = snapshot(frame, '2025-11-01')
test = snapshot(frame, '2026-10-09')
train = train.loc[train['SITUACION_LIMPIA'].isin(TARGET_ORDER)].copy()
test = test.loc[test['SITUACION_LIMPIA'].isin(TARGET_ORDER)].copy()

x_train, x_test, _ = build_features(train, test)
y_train = train['SITUACION_LIMPIA'].map({label: i for i, label in enumerate(TARGET_ORDER)}).to_numpy()
y_test = test['SITUACION_LIMPIA'].map({label: i for i, label in enumerate(TARGET_ORDER)}).to_numpy()
w_train = train['CANTIDAD'].to_numpy()
w_test = test['CANTIDAD'].to_numpy()

print('Entrenamiento:', len(train), 'filas /', w_train.sum(), 'detenidos ponderados')
print('Prueba:', len(test), 'filas /', w_test.sum(), 'detenidos ponderados')
print()
print('Distribución de entrenamiento:')
display(train['SITUACION_LIMPIA'].value_counts().to_frame())
print('Distribución de prueba:')
display(test['SITUACION_LIMPIA'].value_counts().to_frame())


In [ ]:
models = [
    ('Regresión logística ordinal', OrdinalLogisticRegression()),
    ('Random Forest', RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced_subsample',
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )),
]

results = []
for name, model in models:
    metrics, report = evaluate_model(
        name, model, x_train, y_train, w_train, x_test, y_test, w_test
    )
    results.append(metrics)
    print(f'\n{name}')
    display(pd.DataFrame(report).transpose().round(3))

pd.DataFrame(results).round(3)


## 3. Figuras del documento de trabajo

Las figuras se regeneran con `make analysis` y se guardan en `outputs/figures/`.


In [ ]:
from IPython.display import Image, display

for filename in [
    'paises_corte_actual.png',
    'delitos_corte_actual.png',
    'genero_edad_corte_actual.png',
    'stock_por_ano.png',
    'comparacion_paises_2021_2026.png',
]:
    print(filename)
    display(Image(filename=str(ROOT / 'outputs' / 'figures' / filename)))


## 4. Nota de interpretación

El acuerdo es moderado (QWK cercano a 0,40). El modelo es una herramienta exploratoria, no un
instrumento de decisión: faltan variables procesales y longitudinales. Consulta la discusión y las
limitaciones en `reports/informe_actualizado_2026.md`.
